> **Ported notebook.** This notebook was originally a ``midas-ff-pipeline`` walkthrough. ``midas-ff-pipeline`` is deprecated as of 0.4.0; the FF workflow now lives in ``midas-pipeline`` with ``--scan-mode ff``. All API + CLI calls below have been retargeted; behaviour is unchanged. The legacy package will be removed in 1.0.0.


# Smoke walkthrough — synthetic FF-HEDM end-to-end

Generates a 50-grain Au forward simulation and runs the full pipeline
(HKL → peakfit → merge → calc_radius → transforms → cross_det_merge →
binning → indexing → refinement → process_grains) in one cell.

**Edit the paths in cell 2, then *Run All*.** Total runtime: ~2 minutes
on a CPU, ~30 seconds on a GPU.

In [ ]:
from pathlib import Path
import shutil

from midas_pipeline import Pipeline, PipelineConfig, ScanGeometry
from midas_pipeline.testing import generate_synthetic_dataset

## 1. Inputs — edit these

In [ ]:
# Working directory (will be wiped)
work_dir = Path("/tmp/midas_pipeline_smoke")

# Parameter template — points at FF_HEDM/Example/ in the MIDAS source tree.
midas_home = Path.home() / "opt" / "MIDAS"
params_template = midas_home / "FF_HEDM" / "Example" / "Parameters.txt"

n_grains = 50
n_cpus = 8
device = "cpu"        # "cuda", "cuda:0", or "cpu"
dtype = "float64"

## 2. Generate synthetic dataset

In [ ]:
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)

zarr_path = generate_synthetic_dataset(
    out_dir=work_dir / "sim",
    params_template=params_template,
    n_grains=n_grains,
    n_cpus=n_cpus,
)
print(f"synthetic zarr: {zarr_path}")

## 3. Run pipeline

In [ ]:
pipe = Pipeline(
    config=PipelineConfig(
    scan=ScanGeometry.ff(),
        result_dir=str(work_dir / "run"),
        params_file=str(work_dir / "sim" / params_template.name),
        zarr_path=str(zarr_path),
        n_cpus=n_cpus,
        device=device,
        dtype=dtype,
    ),
)
pipe.run()

## 4. Inspect results

In [ ]:
result = pipe.layer_result
print(f"Total grains:    {result.n_grains}")
print(f"Total runtime:   {result.total_duration_s():.1f}s")
print()
for stage in result.all_stage_results():
    print(f"  {stage.stage_name:<20s} {stage.duration_s:7.2f}s")

In [ ]:
df = result.grains_df()
print(df.shape)
df.head()